<div dir="rtl">

# الدرس 59: أسئلة المقابلات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### سوفتماكس مستقرة

</div>

In [ ]:
import numpy as np

def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))      # the largest exponent becomes e^0 = 1
    return e / e.sum(axis=-1, keepdims=True)

with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp([1000.0, 999.0]) / np.exp([1000.0, 999.0]).sum()
print("naive :", naive)
print("stable:", softmax([1000.0, 999.0]).round(4))
assert np.allclose(softmax([[1, 2, 3], [1000, 1000, 1000]]).sum(axis=1), 1)

<div dir="rtl">

### أقرب المتجهات

</div>

In [ ]:
def top_k_cosine(query, matrix, k):
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    q = query / np.linalg.norm(query)
    scores = m @ q
    idx = np.argpartition(-scores, k)[:k]                # the k best, in any order: linear time
    idx = idx[np.argsort(-scores[idx])]                  # then sort only those k
    return idx, scores[idx]

rng = np.random.default_rng(0)
vectors = rng.normal(size=(1000, 32))
query = 3 * vectors[42] + 0.01 * rng.normal(size=32)    # longer than vector 42, same direction
idx, scores = top_k_cosine(query, vectors, 5)
print(idx, scores.round(3))
assert idx[0] == 42

<div dir="rtl">

### الانتباه السببي

</div>

In [ ]:
import torch
import torch.nn.functional as F

def causal_attention(Q, K, V):
    T, d = Q.shape
    scores = Q @ K.T / np.sqrt(d)
    mask = np.tril(np.ones((T, T), dtype=bool))          # token t sees tokens 0..t only
    scores = np.where(mask, scores, -np.inf)
    return softmax(scores) @ V

Q, K, V = rng.normal(size=(3, 6, 8))
out = causal_attention(Q, K, V)
ref = F.scaled_dot_product_attention(*(torch.tensor(x)[None] for x in (Q, K, V)), is_causal=True)[0].numpy()
print("max difference from PyTorch:", np.abs(out - ref).max())

V_future = V.copy()
V_future[-1] += 100                                      # change only the last token
assert np.allclose(causal_attention(Q, K, V_future)[:-1], out[:-1])   # earlier tokens cannot see it

<div dir="rtl">

### الدقة الإيجابية والاستدعاء بلا مكتبات

</div>

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def precision_recall_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    p = tp / (tp + fp) if tp + fp else 0.0               # no positive predictions
    r = tp / (tp + fn) if tp + fn else 0.0               # no positive examples
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

y_true, y_pred = rng.integers(0, 2, 200), rng.integers(0, 2, 200)
ours = precision_recall_f1(y_true, y_pred)
theirs = precision_recall_fscore_support(y_true, y_pred, average="binary")[:3]
print([round(float(x), 4) for x in ours], [round(float(x), 4) for x in theirs])
assert np.allclose(ours, theirs) and precision_recall_f1([0, 0], [0, 0]) == (0.0, 0.0, 0.0)

<div dir="rtl">

## تمرين: احسب معاملات كتلة ترانسفورمر

سؤال مقابلات شائع: كتلة ترانسفورمر كما في الدرس 48. البُعد $d = 512$، والشبكة الأمامية أعرض أربع مرات، وكل الطبقات الخطية بانحياز، وطبقتا تطبيع.

1. احسب بيدك معاملات الانتباه، ثم الشبكة الأمامية، ثم التطبيع.
2. اكتب القانون العام بدلالة $d$.
3. قارن بالقاعدة التقريبية المشهورة: $12 \, d^2$. وتحقق بباي تورش.

</div>

In [ ]:
# اكتب حلّك هنا
